In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from pathlib import Path
import pandas as pd

data_dir = Path("BankData")

# Collect unique files only
csv_files = sorted({f.resolve(): f for f in data_dir.iterdir() if f.suffix.lower() == ".csv"}.values())

for file in csv_files:
    print(f"\n{'='*25} {file.name} {'='*25}")
    try:
        df = pd.read_csv(file, sep=';', nrows=3)
        display(df)
    except Exception as e:
        print(f"Error reading {file.name}: {e}")


========================= account.csv =========================


,account_id,district_id,frequency,date
0,576,55,POPLATEK MESICNE,930101
1,3818,74,POPLATEK MESICNE,930101
2,704,55,POPLATEK MESICNE,930101



========================= card.csv =========================


,card_id,disp_id,type,issued
0,1005,9285,classic,931107 00:00:00
1,104,588,classic,940119 00:00:00
2,747,4915,classic,940205 00:00:00



========================= client.csv =========================


...

,loan_id,account_id,date,amount,duration,payments,status
0,5314,1787,930705,96396,12,8033.0,B
1,5316,1801,930711,165960,36,4610.0,A
2,6863,9188,930728,127080,60,2118.0,A



========================= order.csv =========================


,order_id,account_id,bank_to,account_to,amount,k_symbol
0,29401,1,YZ,87144583,2452.0,SIPO
1,29402,2,ST,89597016,3372.7,UVER
2,29403,2,QR,13943797,7266.0,SIPO



========================= trans.csv =========================


,trans_id,account_id,date,type,operation,amount,balance,k_symbol,bank,account
0,695247,2378,930101,PRIJEM,VKLAD,700.0,700.0,NaN,NaN,NaN
1,171812,576,930101,PRIJEM,VKLAD,900.0,900.0,NaN,NaN,NaN
2,207264,704,930101,PRIJEM,VKLAD,1000.0,1000.0,NaN,NaN,NaN


In [3]:
# Load the full loan table
loan = pd.read_csv(data_dir / "loan.csv", sep=";")

print(f"Total loan records: {len(loan)}")
print(f"Unique account IDs: {loan['account_id'].nunique()}")

print("\nCounts per loan status:")
print(loan["status"].value_counts())

print("\nPercentage breakdown:")
print((loan["status"].value_counts(normalize=True) * 100).round(2))

Total loan records: 682
Unique account IDs: 682

Counts per loan status:
status
C    403
A    203
D     45
B     31
Name: count, dtype: int64

Percentage breakdown:
status
C    59.09
A    29.77
D     6.60
B     4.55
Name: proportion, dtype: float64


In [4]:
# 1. Merge transactions with loan grant dates
trans = pd.read_csv(data_dir / "trans.csv", sep=";")
loan_dates = loan[['account_id', 'date']].rename(columns={'date': 'loan_date'})
trans_loans = trans.merge(loan_dates, on='account_id', how='inner')

# 2. Filter strictly for transactions BEFORE the loan date
pre_loan_trans = trans_loans[trans_loans['date'] < trans_loans['loan_date']]

# 3. Check for pre-loan card withdrawals
card_ops = pre_loan_trans[pre_loan_trans['operation'] == 'VYBER KARTOU']
print(f"Unique borrowers with pre-loan card activity: {card_ops['account_id'].nunique()} out of 682")

C:\Users\omare\AppData\Local\Temp\ipykernel_22984\1591252777.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  trans = pd.read_csv(data_dir / "trans.csv", sep=";")


Unique borrowers with pre-loan card activity: 24 out of 682
